# 01 · Outdoor E-vs-S GLMs and window aggregation

Builds every window table the downstream notebooks read.

**Outdoor models (per SNP, quasibinomial, S = reference so every coefficient reads E vs S)**

| Track tag | Model | Data used | Estimand |
|---|---|---|---|
| `tpt<k>` | `AF ~ treat.fix` | samples from timepoint *k* only | E-S logit difference at timepoint *k* (own dispersion, own df) |
| `pooled` (**primary**) | `AF ~ treat.fix + tpt` | all timepoints | E-S logit difference within timepoint, averaged over timepoints |

Per SNP: two-sided p → |z| = `qnorm(p/2, lower = FALSE)` (p floored at 1e-300).

**Windows.** Outdoor SNP |z| is summarised by mean / median / max at 50, 100 and 500 kb, computed directly from the SNPs at every scale. xQTL panels come as 50 kb window scans: one scan row per 50 kb grid window → BH across the non-overlapping windows within each panel → |z| from the BH-adjusted p. At 100 / 500 kb the xQTL value is the median of the 50 kb window z's inside the larger window.

**Downstream default:** primary track `pooled`, window stat `median`, 50 kb. Thresholds everywhere = each track's own 95th percentile.

### Inputs → outputs

| In (read only, `Final_Data_Inputs/`) | Out (`Final_Final_Data/`) |
|---|---|
| `Outdoor/rudflies_2023_hafpipe.csv` | `Data/01_GLM/rudflies_2023_SvE_GLM_SNP.tsv` (SNP-level b / se / p / q / \|z\| / phi per track, per-tpt dAF) |
| `Outdoor/rudflies_2023_meta.tsv` | `Data/01_GLM/conc_50kb.csv`, `conc_100kb.csv`, `conc_500kb.csv` |
| `xQTL/Spino3_DGRP/Spino3.pseudoscan.chr*.txt` | `Data/01_GLM/window_vs_nsnp_spearman.csv` |
| `xQTL/Spino3_DSPR/Step_1kb_Window_50kb.pseudoscan.chr*.txt` | `Figures/01_GLM/SvE_SNP_manhattans.png` |
| `xQTL/Spino4_DSPR/Spino4.pseudoscan.chr*.txt` | `Figures/01_GLM/window_manhattan_<scale>_<stat>.png` (9 files) |
| `xQTL/Spino3Out_DSPR/Spino3Out.pseudoscan.chr*.txt` | |

## 1. Setup

In [5]:
ROOT    <- "D:/xQTL_2025_Data/Final_Window_Analysis/Analysis_jupyter_notebooks/Final_Notebooks"
IN_DIR  <- file.path(ROOT, "Final_Data_Inputs")        # raw inputs, never written to
OUT_DIR <- file.path(ROOT, "Final_Final_Data")         # everything this pipeline writes
DATA    <- file.path(OUT_DIR, "Data",    "01_GLM")
FIGS    <- file.path(OUT_DIR, "Figures", "01_GLM")
CHUNKS  <- file.path(DATA, "glm_chunks")
for (d in c(DATA, FIGS, CHUNKS)) dir.create(d, recursive = TRUE, showWarnings = FALSE)
setwd(OUT_DIR)

## ---- analysis constants ------------------------------------------------
KEEP_SAMP    <- setdiff(1:171, c(10:29, 90))   # 10-29 excluded as before; 90 = cage 29 tpt 2 (dropped)
MIN_MAF      <- 0.05                            # folded mean AF across retained samples
PRIMARY      <- "pooled"                        # AF ~ treat.fix + tpt
WIN          <- 50000                           # base window = xQTL scan window
SCALES       <- c(50000, 100000, 500000)
STATS        <- c("mean", "median", "max")
POS_IS       <- "center"                         # xQTL scan `pos` = window start ("start") or centre ("center")
TOL          <- 2500                            # max bp between a grid point and the scan row used for it
XQTL_Z_FROM  <- "fdr"                           # "fdr": |z| from BH-adjusted window p; "raw": from raw window p
Q_CUT        <- 0.95
CHROMS       <- c("X", "2L", "2R", "3L", "3R")
N_CORES      <- 30L

suppressPackageStartupMessages({
  library(data.table); library(ggplot2); library(doSNOW); library(foreach)
})

p_to_z    <- function(p) qnorm(pmax(p, 1e-300) / 2, lower.tail = FALSE)   # two-sided p -> |z|
strip_chr <- function(x) sub("^chr", "", x)

## 2. Outdoor allele frequencies, sample filter, MAF filter

One genome-wide MAF filter, so every track is fit on the same SNP set.

In [6]:
outdoor_AF <- fread(file.path(IN_DIR, "Outdoor/rudflies_2023_hafpipe.csv"))
haf.meta   <- fread(file.path(IN_DIR, "Outdoor/rudflies_2023_meta.tsv"), header = TRUE)
setDF(outdoor_AF); setDF(haf.meta)

names(outdoor_AF) <- gsub("X", "", gsub("[.]af", "", names(outdoor_AF)))
names(outdoor_AF)[1:2] <- c("CHROM", "POS")

meta <- haf.meta[haf.meta$batch == "a" &
                 haf.meta$treat.fix %in% c("E", "S") &
                 haf.meta$samp %in% KEEP_SAMP, ]
meta <- meta[as.character(meta$samp) %in% names(outdoor_AF), ]
meta$treat.fix <- factor(meta$treat.fix, levels = c("S", "E"))
meta$tpt       <- factor(meta$tpt)
TPTS <- levels(meta$tpt)
print(table(treat = meta$treat.fix, tpt = meta$tpt))
stopifnot(all(table(meta$treat.fix, meta$tpt) > 1))   # every tpt needs >1 E and >1 S sample

af.mat <- as.matrix(outdoor_AF[, as.character(meta$samp)])
loci   <- outdoor_AF[, c("CHROM", "POS")]
rm(outdoor_AF); invisible(gc())

af.mean <- rowMeans(af.mat)                              # NA if any sample NA
keep    <- !is.na(af.mean) & pmin(af.mean, 1 - af.mean) >= MIN_MAF
cat(sprintf("NA + MAF filter: kept %d / %d SNPs (folded mean AF >= %.2f)\n",
            sum(keep), length(keep), MIN_MAF))
af.mat  <- af.mat[keep, , drop = FALSE]
loci    <- loci[keep, ]
af.mean <- af.mean[keep]
rm(keep); invisible(gc())

     tpt
treat  1  2  3  4
    S 10  5  5  5
    E 10  8  9  9
NA + MAF filter: kept 1772618 / 3231230 SNPs (folded mean AF >= 0.05)


## 3. Per-SNP GLMs (per-timepoint subsets + pooled)

Fits use `glm.fit` with the `summary.snp_glm` dispersion / SE math, so results are identical to `summary(glm(..., family = quasibinomial))`. Batched PSOCK workers, chunks written to disk to bound RAM. The chunk folder is cleared first so stale chunks from an older run can never be bound in.

In [7]:
TRACK_T <- paste0("tpt", TPTS)
TRACKS  <- c(TRACK_T, PRIMARY)

## design matrices, built once
X_pool <- model.matrix(~ treat.fix + tpt, data = meta)
idx_t  <- lapply(TPTS, function(t) which(meta$tpt == t))
X_t    <- lapply(idx_t, function(ix) model.matrix(~ treat.fix, data = droplevels(meta[ix, ])))
isE_t  <- lapply(TPTS, function(t) meta$tpt == t & meta$treat.fix == "E")
isS_t  <- lapply(TPTS, function(t) meta$tpt == t & meta$treat.fix == "S")

## quasibinomial fit: glm.fit + summary.snp_glm math
qb_fit <- function(X, y) {
  f   <- suppressWarnings(glm.fit(X, y, family = quasibinomial()))
  r   <- seq_len(f$rank)
  phi <- sum((f$weights * f$residuals^2)[f$weights > 0]) / f$df.residual
  se  <- setNames(rep(NA_real_, ncol(X)), colnames(X))
  se[f$qr$pivot[r]] <- sqrt(diag(chol2inv(f$qr$qr[r, r, drop = FALSE])) * phi)
  b   <- f$coefficients
  p   <- 2 * pt(abs(b / se), f$df.residual, lower.tail = FALSE)
  list(b = b, se = se, p = p, phi = phi)
}

OUT_NAMES <- c(unlist(lapply(TRACKS, function(tg) paste0(c("b_", "se_", "p_", "phi_"), tg))),
               paste0("dAF_t", TPTS))
NOUT <- length(OUT_NAMES)

snp_worker <- function(y) {
  y   <- as.numeric(y)
  out <- setNames(rep(NA_real_, NOUT), OUT_NAMES)
  for (k in seq_along(TPTS)) {
    tg <- TRACK_T[k]
    f  <- tryCatch(qb_fit(X_t[[k]], y[idx_t[[k]]]), error = function(e) NULL)
    if (!is.null(f))
      out[paste0(c("b_", "se_", "p_", "phi_"), tg)] <-
        c(f$b[["treat.fixE"]], f$se[["treat.fixE"]], f$p[["treat.fixE"]], f$phi)
    out[[paste0("dAF_t", TPTS[k])]] <- mean(y[isE_t[[k]]]) - mean(y[isS_t[[k]]])
  }
  f <- tryCatch(qb_fit(X_pool, y), error = function(e) NULL)
  if (!is.null(f))
    out[paste0(c("b_", "se_", "p_", "phi_"), PRIMARY)] <-
      c(f$b[["treat.fixE"]], f$se[["treat.fixE"]], f$p[["treat.fixE"]], f$phi)
  out
}

## ---- cluster + chunked run ---------------------------------------------
unlink(list.files(CHUNKS, full.names = TRUE))
cl <- parallel::makeCluster(N_CORES, type = "PSOCK")
registerDoSNOW(cl)
parallel::clusterExport(cl, c("X_pool", "X_t", "idx_t", "isE_t", "isS_t", "TPTS", "TRACK_T",
                              "PRIMARY", "OUT_NAMES", "NOUT", "qb_fit", "snp_worker"))

nsnp   <- nrow(af.mat)
CHUNK  <- 100000L
BATCH  <- 500L
pb <- txtProgressBar(max = nsnp, style = 3)
for (st in seq(1L, nsnp, by = CHUNK)) {
  en  <- min(st + CHUNK - 1L, nsnp)
  blk <- af.mat[st:en, , drop = FALSE]
  batch.idx <- split(seq_len(nrow(blk)), ceiling(seq_len(nrow(blk)) / BATCH))
  opts <- list(progress = function(n) setTxtProgressBar(pb, min(st - 1L + n * BATCH, en)))
  pmat <- foreach(ix = batch.idx, .combine = rbind, .packages = "stats",
                  .options.snow = opts) %dopar% {
    t(vapply(ix, function(j) snp_worker(blk[j, ]), numeric(NOUT)))
  }
  rownames(pmat) <- NULL; colnames(pmat) <- OUT_NAMES
  saveRDS(cbind(loci[st:en, ], af_mean = af.mean[st:en], pmat),
          file.path(CHUNKS, sprintf("chunk_%09d.rds", st)))
  rm(blk, pmat); invisible(gc(FALSE))
}
close(pb)
parallel::stopCluster(cl)

  |======================================================================| 100%


## 4. Bind, BH + |z| per track, write the SNP table

In [8]:
snp_glm <- rbindlist(lapply(sort(list.files(CHUNKS, full.names = TRUE)), readRDS))
for (tg in TRACKS) {
  set(snp_glm, j = paste0("q_",    tg), value = p.adjust(snp_glm[[paste0("p_", tg)]], method = "BH"))
  set(snp_glm, j = paste0("absz_", tg), value = p_to_z(snp_glm[[paste0("p_", tg)]]))
}
fwrite(snp_glm, file.path(DATA, "rudflies_2023_SvE_GLM_SNP.tsv"), sep = "\t", nThread = N_CORES)

cat("SNPs:", nrow(snp_glm), "\n")
for (tg in TRACKS)
  cat(sprintf("  %-8s NA: %7d  p<0.05: %.3f  q<0.05: %7d  mean|z|: %.3f (null ~0.80)  median phi: %.3f\n",
              tg, sum(is.na(snp_glm[[paste0("p_", tg)]])),
              mean(snp_glm[[paste0("p_", tg)]] < 0.05, na.rm = TRUE),
              sum(snp_glm[[paste0("q_", tg)]] < 0.05, na.rm = TRUE),
              mean(snp_glm[[paste0("absz_", tg)]], na.rm = TRUE),
              median(snp_glm[[paste0("phi_", tg)]], na.rm = TRUE)))

SNPs: 1772618 
  tpt1     NA:       0  p<0.05: 0.080  q<0.05:       0  mean|z|: 0.897 (null ~0.80)  median phi: 0.015
  tpt2     NA:       0  p<0.05: 0.169  q<0.05:      39  mean|z|: 1.132 (null ~0.80)  median phi: 0.009
  tpt3     NA:       0  p<0.05: 0.276  q<0.05:  194740  mean|z|: 1.404 (null ~0.80)  median phi: 0.006
  tpt4     NA:       0  p<0.05: 0.143  q<0.05:       6  mean|z|: 1.086 (null ~0.80)  median phi: 0.022
  pooled   NA:       0  p<0.05: 0.318  q<0.05:  322479  mean|z|: 1.543 (null ~0.80)  median phi: 0.014


## 5. QC: SNP-level Manhattans, one panel per track

y = -log10(p); dashed red = BH q < 0.05 for that track. Non-significant SNPs thinned for plotting speed (every SNP with p < 0.01 kept).

In [9]:
track_lab <- function(tg) ifelse(tg == PRIMARY, "Pooled: AF ~ treat + tpt",
                                 paste0("Timepoint ", sub("^tpt", "", tg), ": AF ~ treat"))
THIN_P <- 0.01; THIN_FRAC <- 0.02
CHR_PLOT <- c("2L", "2R", "3L", "3R", "X")

d <- snp_glm[, c(list(CHROM = strip_chr(as.character(CHROM)), POS = POS),
             .SD), .SDcols = c(paste0("p_", TRACKS), paste0("q_", TRACKS))]
d <- d[CHROM %in% CHR_PLOT]
d[, CHROM := factor(CHROM, levels = CHR_PLOT)]
setorder(d, CHROM, POS)
off <- d[, .(len = as.numeric(max(POS))), by = CHROM][, off := c(0, cumsum(len)[-.N])]
d[off, gpos := POS + i.off, on = "CHROM"]
axis_df <- d[, .(mid = (min(gpos) + max(gpos)) / 2), by = CHROM]

thr <- rbindlist(lapply(TRACKS, function(tg) {
  ps <- d[[paste0("p_", tg)]][d[[paste0("q_", tg)]] < 0.05 & !is.na(d[[paste0("q_", tg)]])]
  data.table(track = tg, thr = if (length(ps)) -log10(max(ps)) else NA_real_)
}))
long <- melt(d[, c("CHROM", "gpos", paste0("p_", TRACKS)), with = FALSE],
             id.vars = c("CHROM", "gpos"), variable.name = "track", value.name = "p", na.rm = TRUE)
long[, track := sub("^p_", "", track)]
set.seed(1)
long <- long[p < THIN_P | runif(.N) < THIN_FRAC][, mlp := -log10(p)]
long[, track := factor(track, levels = TRACKS, labels = track_lab(TRACKS))]
thr  <- thr[!is.na(thr)][, track := factor(track, levels = TRACKS, labels = track_lab(TRACKS))]

g <- ggplot(long, aes(gpos, mlp, colour = CHROM)) +
  geom_point(size = 0.25, stroke = 0) +
  geom_hline(data = thr, aes(yintercept = thr), linetype = "dashed", colour = "red", linewidth = 0.4) +
  facet_wrap(~ track, ncol = 1, scales = "free_y") +
  scale_colour_manual(values = setNames(rep(c("grey25", "grey65"), length.out = 5), CHR_PLOT), guide = "none") +
  scale_x_continuous(breaks = axis_df$mid, labels = axis_df$CHROM, expand = expansion(mult = 0.005)) +
  labs(x = NULL, y = expression(-log[10](p))) +
  theme_classic(base_size = 10) +
  theme(strip.background = element_blank(), strip.text = element_text(hjust = 0, face = "bold"))
ggsave(file.path(FIGS, "SvE_SNP_manhattans.png"), g,
       width = 12, height = 1.6 * length(TRACKS) + 0.5, dpi = 300, limitsize = FALSE)
rm(d, long); invisible(gc())

## 6. xQTL panels on a shared 50 kb grid

Grid per chromosome: from the first position shared by all four scans, stepping 50 kb. Each grid point is one non-overlapping 50 kb scan window; the scan row nearest each grid point (within `TOL`) supplies its Wald p. BH is applied across these non-overlapping windows within each panel, and the window |z| comes from the BH-adjusted p (`XQTL_Z_FROM = "fdr"`). Raw p, q and raw-p z are kept in the 50 kb table.

In [10]:
SCANS <- c(dgrp      = "xQTL/Spino3_DGRP/Spino3.pseudoscan.chr%s.txt",
           dspr3     = "xQTL/Spino3_DSPR/Step_1kb_Window_50kb.pseudoscan.chr%s.txt",
           dspr4     = "xQTL/Spino4_DSPR/Spino4.pseudoscan.chr%s.txt",
           outspino3 = "xQTL/Spino3Out_DSPR/Spino3Out.pseudoscan.chr%s.txt")
PNAMES <- names(SCANS)

## read.table (not fread): the scan files carry row names, which read.table detects
scans <- lapply(SCANS, function(fmt) {
  d <- rbindlist(lapply(CHROMS, function(ch)
         as.data.table(read.table(file.path(IN_DIR, sprintf(fmt, ch))))), fill = TRUE)
  d[is.finite(Wald_log10p),
    .(chr = strip_chr(as.character(chr)), pos = as.numeric(pos),
      p = 10^(-pmin(Wald_log10p, 300)), cM)]
})

grid <- rbindlist(lapply(scans, function(d) d[, .(lo = min(pos), hi = max(pos)), by = chr])
                  )[, .(lo = max(lo), hi = min(hi)), by = chr]
grid <- grid[, .(pos = seq(lo, hi, by = WIN)), by = chr]
span <- grid[, .(start = min(pos), end = max(pos)), by = chr]
cat("grid windows per chr:\n"); print(grid[, .N, by = chr])

prep_xqtl <- function(d, pname) {
  m <- d[grid, on = .(chr, pos), roll = "nearest", .(chr, pos, p, cM, scan_pos = x.pos)]
  m <- m[abs(scan_pos - pos) <= TOL][, scan_pos := NULL]
  cat(sprintf("  %-10s %d / %d grid windows matched\n", pname, nrow(m), nrow(grid)))
  m[, q := p.adjust(p, method = "BH")]               # BH across non-overlapping windows
  m[, z_raw := p_to_z(p)]
  m[, z := if (XQTL_Z_FROM == "fdr") p_to_z(q) else z_raw]
  if (pname != "dspr3") m[, cM := NULL]               # keep one genetic map (Spino3 DSPR)
  setnames(m, c("p", "q", "z", "z_raw"), paste0(c("p_", "q_", "z_", "zraw_"), pname))
  m
}
cat("xQTL rows on grid:\n")
xq <- Reduce(function(a, b) merge(a, b, by = c("chr", "pos")), Map(prep_xqtl, scans, PNAMES))
xq <- merge(xq, span[, .(chr, start)], by = "chr")
cat("grid windows with all four panels:", nrow(xq), "\n")
cat("windows with BH q < 0.05:\n")
for (pn in PNAMES) cat(sprintf("  %-10s %d\n", pn, sum(xq[[paste0("q_", pn)]] < 0.05)))

grid windows per chr:
      chr     N
   <char> <int>
1:      X   399
2:     2L   446
3:     2R   416
4:     3L   543
5:     3R   586
xQTL rows on grid:
  dgrp       2390 / 2390 grid windows matched
  dspr3      2078 / 2390 grid windows matched
  dspr4      2079 / 2390 grid windows matched
  outspino3  2079 / 2390 grid windows matched
grid windows with all four panels: 2078 
windows with BH q < 0.05:
  dgrp       1505
  dspr3      1002
  dspr4      1161
  outspino3  507


## 7. Window tables at 50 / 100 / 500 kb

Every outdoor statistic is computed from the SNPs falling in the window at that scale (no median-of-medians). Larger windows are blocks of consecutive 50 kb grid tiles aligned to the grid start. xQTL at each scale = median of the 50 kb window z's in the block (at 50 kb that is the window z itself). Blocks with fewer than half their tiles present in all four xQTL panels are dropped. No windows are dropped for falling outside a full 500 kb region (the old 10-window region filter silently lost chromosome ends).

In [11]:
off_bp <- if (POS_IS == "center") WIN / 2 else 0
SNP_Z  <- paste0("absz_", TRACKS)
DAF    <- paste0("dAF_t", TPTS)
XZ     <- paste0("z_", PNAMES)

## outdoor SNPs -> 50 kb grid tile
o <- snp_glm[, c(list(chr = strip_chr(as.character(CHROM)), POS = as.numeric(POS)), .SD),
         .SDcols = c(SNP_Z, DAF)]
o <- merge(o, span, by = "chr")
o[, tile := start + floor((POS - start + off_bp) / WIN) * WIN]
o <- o[tile >= start & tile <= end][, c("POS", "end") := NULL]

na_inf <- function(dt, cols) for (cc in cols) set(dt, which(!is.finite(dt[[cc]])), cc, NA_real_)

agg_scale <- function(W) {
  key <- c("chr", "bw")

  ## outdoor: SNP-level summaries per track
  o[, bw := (tile - start) %/% W]
  res <- o[, .(n_snp = .N), by = key]
  for (s in STATS) {
    f <- switch(s, mean = mean, median = median, max = max)
    a <- suppressWarnings(o[, lapply(.SD, f, na.rm = TRUE), by = key, .SDcols = SNP_Z])
    setnames(a, SNP_Z, paste0("out_", TRACKS, "_", s))
    res <- merge(res, a, by = key)
  }
  a1 <- o[, lapply(.SD, mean, na.rm = TRUE), by = key, .SDcols = DAF]
  setnames(a1, DAF, paste0("mean_", DAF))
  a2 <- o[, lapply(.SD, function(v) mean(abs(v), na.rm = TRUE)), by = key, .SDcols = DAF]
  setnames(a2, DAF, paste0("meanabs_", DAF))
  res <- merge(merge(res, a1, by = key), a2, by = key)

  ## xQTL: median of the 50 kb window z's in the block
  xq[, bw := (pos - start) %/% W]
  xb <- xq[, c(list(start = start[1], n_tiles = .N, cM = mean(cM, na.rm = TRUE)),
               lapply(.SD, median, na.rm = TRUE)), by = key, .SDcols = XZ]
  if (W == WIN)                                    # 50 kb: also carry window p / q / raw-p z
    xb <- merge(xb, xq[, c(key, paste0("p_", PNAMES), paste0("q_", PNAMES), paste0("zraw_", PNAMES)),
                       with = FALSE], by = key)
  xb <- xb[n_tiles >= ceiling(0.5 * W / WIN)]

  b <- merge(res, xb, by = key)
  b[, win_start := start + bw * W - off_bp][, win_end := win_start + W]
  na_inf(b, setdiff(names(b), c("chr", "bw")))
  b <- b[complete.cases(b[, c(paste0("out_", PRIMARY, "_median"), XZ), with = FALSE])]
  b[, chr := factor(chr, levels = CHROMS)]
  setorder(b, chr, win_start)
  b[, chr := as.character(chr)]

  cols <- c("chr", "win_start", "win_end", "cM", "n_tiles", "n_snp", XZ,
            if (W == WIN) c(paste0("p_", PNAMES), paste0("q_", PNAMES), paste0("zraw_", PNAMES)),
            unlist(lapply(TRACKS, function(tg) paste0("out_", tg, "_", STATS))),
            paste0("mean_", DAF), paste0("meanabs_", DAF))
  b <- b[, ..cols]
  f <- file.path(DATA, sprintf("conc_%dkb.csv", W / 1000))
  fwrite(b, f)
  cat(sprintf("wrote %s: %d windows, median %d SNPs/window\n", basename(f), nrow(b),
              as.integer(median(b$n_snp))))
  b
}

conc <- setNames(lapply(SCALES, agg_scale), paste0(SCALES / 1000, "kb"))

wrote conc_50kb.csv: 1951 windows, median 848 SNPs/window
wrote conc_100kb.csv: 1002 windows, median 1711 SNPs/window
wrote conc_500kb.csv: 205 windows, median 8750 SNPs/window


## 8. Sanity: window scores vs SNP count

Window |z| should not simply track SNP density. Max will (more SNPs, more chances for a big value), which is why median is the downstream default.

In [12]:
nsnp_rho <- rbindlist(lapply(names(conc), function(sc) {
  b <- conc[[sc]]
  rbindlist(lapply(TRACKS, function(tg) rbindlist(lapply(STATS, function(s) {
    cc <- paste0("out_", tg, "_", s)
    data.table(scale = sc, track = tg, stat = s,
               rho_vs_nsnp = cor(b[[cc]], b$n_snp, method = "spearman", use = "complete.obs"))
  }))))
}))
fwrite(nsnp_rho, file.path(DATA, "window_vs_nsnp_spearman.csv"))
print(dcast(nsnp_rho, track + stat ~ scale, value.var = "rho_vs_nsnp"), digits = 3)

Key: <track, stat>
     track   stat   100kb   500kb    50kb
    <char> <char>   <num>   <num>   <num>
 1: pooled    max  0.0503  0.0382  0.0743
 2: pooled   mean -0.1248 -0.1520 -0.1240
 3: pooled median -0.1189 -0.1422 -0.1310
 4:   tpt1    max  0.2661  0.3216  0.2617
 5:   tpt1   mean -0.1672 -0.1702 -0.1559
 6:   tpt1 median -0.1781 -0.1688 -0.1690
 7:   tpt2    max  0.2034  0.2429  0.2189
 8:   tpt2   mean -0.1853 -0.2532 -0.1638
 9:   tpt2 median -0.1842 -0.2539 -0.1682
10:   tpt3    max  0.2208  0.2135  0.2367
11:   tpt3   mean -0.0845 -0.1244 -0.0661
12:   tpt3 median -0.0814 -0.1291 -0.0636
13:   tpt4    max  0.1720  0.1601  0.2024
14:   tpt4   mean -0.1178 -0.1463 -0.1287
15:   tpt4 median -0.1078 -0.1459 -0.1274


## 9. Window Manhattans: 3 scales × 3 stats

One figure per (scale, stat): every outdoor track plus the four xQTL panels. Dashed red = each panel's own genome-wide 95th percentile (the significance rule used downstream).

In [13]:
PANEL_LAB <- c(dgrp = "LC50-DGRP-I", dspr3 = "LC50-DSPR-I", dspr4 = "LC90-DSPR-I", outspino3 = "LC50-DSPR-O")

plot_window_manhattan <- function(sc, s) {
  d <- copy(conc[[sc]])
  d <- d[chr %in% CHR_PLOT][, chr := factor(chr, levels = CHR_PLOT)]
  setorder(d, chr, win_start)
  off <- d[, .(len = as.numeric(max(win_end))), by = chr][, off := c(0, cumsum(len)[-.N])]
  d[off, gpos := win_start + i.off, on = "chr"]
  ax <- d[, .(mid = (min(gpos) + max(gpos)) / 2), by = chr]

  cols <- c(paste0("out_", TRACKS, "_", s), XZ)
  labs <- setNames(c(paste("Outdoor", track_lab(TRACKS)),
                     paste0("xQTL ", PANEL_LAB[PNAMES], if (sc != "50kb") " (median of 50kb)")), cols)
  long <- melt(d[, c("chr", "gpos", cols), with = FALSE], id.vars = c("chr", "gpos"),
               variable.name = "track", value.name = "score", na.rm = TRUE)
  thr  <- long[, .(thr = quantile(score, Q_CUT)), by = track]
  long[, track := factor(track, levels = cols, labels = labs[cols])]
  thr[,  track := factor(track, levels = cols, labels = labs[cols])]

  g <- ggplot(long, aes(gpos, score, colour = chr)) +
    geom_point(size = 0.6, stroke = 0) +
    geom_hline(data = thr, aes(yintercept = thr), linetype = "dashed", colour = "red", linewidth = 0.35) +
    facet_wrap(~ track, ncol = 1, scales = "free_y") +
    scale_colour_manual(values = setNames(rep(c("grey20", "grey60"), length.out = 5), CHR_PLOT), guide = "none") +
    scale_x_continuous(breaks = ax$mid, labels = ax$chr, expand = expansion(mult = 0.005)) +
    labs(x = NULL, y = "window |z|",
         title = sprintf("%s windows, outdoor %s |z|  (dashed = genome-wide %g%% quantile)",
                         sc, s, 100 * Q_CUT)) +
    theme_classic(base_size = 10) +
    theme(strip.background = element_blank(), strip.text = element_text(hjust = 0, face = "bold"),
          plot.title = element_text(size = 10))
  fn <- file.path(FIGS, sprintf("window_manhattan_%s_%s.png", sc, s))
  ggsave(fn, g, width = 12, height = 1.3 * length(cols) + 0.8, dpi = 300, limitsize = FALSE)
  cat("wrote", basename(fn), "\n")
}
for (sc in names(conc)) for (s in STATS) plot_window_manhattan(sc, s)

wrote window_manhattan_50kb_mean.png 
wrote window_manhattan_50kb_median.png 
wrote window_manhattan_50kb_max.png 
wrote window_manhattan_100kb_mean.png 
wrote window_manhattan_100kb_median.png 
wrote window_manhattan_100kb_max.png 
wrote window_manhattan_500kb_mean.png 
wrote window_manhattan_500kb_median.png 
wrote window_manhattan_500kb_max.png 
